Line A : Gare du nord-chatelet-Saint-michel

Line B : Louvre-chatelet-Marais 

In [1]:
mini_graph = {
    # --- STATION HUBS (The entry/exit points) ---
    "Gare du Nord": [(("Gare du Nord", "A"), 0)],
    "Saint-Michel": [(("Saint-Michel", "A"), 0)],
    "Louvre":       [(("Louvre", "B"), 0)],
    "Marais":       [(("Marais", "B"), 0)],
    "Châtelet":     [(("Châtelet", "A"), 2), (("Châtelet", "B"), 5)], 

    # --- PLATFORM NODES (Where the riding and transferring happens) ---
    
    # Line A Platforms
    ("Gare du Nord", "A"): [
        ("Gare du Nord", 0),          # Exit to Hub 
        (("Châtelet", "A"), 4)        # Ride to Châtelet 
    ],
    
    ("Châtelet", "A"): [
        ("Châtelet", 0),              # Exit to Hub 
        (("Gare du Nord", "A"), 4),   # Ride to Gare du Nord 
        (("Saint-Michel", "A"), 2),   # Ride to Saint-Michel 
        (("Châtelet", "B"), 5)        # Transfer to Line B 
    ],
    
    ("Saint-Michel", "A"): [
        ("Saint-Michel", 0),          # Exit 
        (("Châtelet", "A"), 2)        # Ride 
    ],

    # Line B Platforms
    ("Louvre", "B"): [
        ("Louvre", 0),                # Exit 
        (("Châtelet", "B"), 3)        # Ride 
    ],
    
    ("Châtelet", "B"): [
        ("Châtelet", 0),              # Exit 
        (("Louvre", "B"), 3),         # Ride 
        (("Marais", "B"), 3),         # Ride 
        (("Châtelet", "A"), 5)        # Transfer to Line A 
    ],
    
    ("Marais", "B"): [
        ("Marais", 0),                # Exit 
        (("Châtelet", "B"), 3)        # Ride 
    ]
}

# The physical coordinates of each station hub
STATION_COORDS = {
    "Gare du Nord": (2.0, 5.0),
    "Châtelet":     (2.0, 2.0),
    "Saint-Michel": (2.0, 0.0), #### 1min = 1 unit distance
    "Louvre":       (0.0, 2.0),
    "Marais":       (4.0, 2.0)

    
}


Dijkstra Algorithm

In [2]:
import heapq
from itertools import count # Add this import at the top

import math

def dijkstra_with_path(graph, source, target):
    # Initialize distances and parent tracking
    distances = {node: float('inf') for node in graph}
    distances[source] = 0
    parents = {node: None for node in graph}
    
    # Initialize the counter to break ties in the priority queue
    tie_breaker = count() 
    
    # Heap element format: (cumulative_time, tie_breaker_value, node)
    pq = [(0, next(tie_breaker), source)]
    
    while pq:
        # Pop the node with the smallest cumulative time
        current_time, _, current_node = heapq.heappop(pq)
        
        # Target reached; we can stop early
        if current_node == target:
            break
            
        # Skip processing if we found a shorter path to this node already
        if current_time > distances[current_node]:
            continue
            
        # Explore neighbors
        for neighbor, travel_time in graph[current_node]:
            new_time = current_time + travel_time
            
            # If a shorter path to the neighbor is found
            if new_time < distances[neighbor]:
                distances[neighbor] = new_time
                parents[neighbor] = current_node
                heapq.heappush(pq, (new_time, next(tie_breaker), neighbor))
                
    # --- Path Reconstruction ---
    path = []
    # Only reconstruct if the target was reached (distance is not infinity)
    if distances[target] != float('inf'):
        current = target
        while current is not None:
            path.append(current)
            current = parents[current]
        path.reverse()  # Reverse to get the path from source to target
        
    return path, distances[target]

In [3]:
# TEST DIJKSTRA


dijkstra_with_path(mini_graph, "Saint-Michel", "Louvre")

(['Saint-Michel',
  ('Saint-Michel', 'A'),
  ('Châtelet', 'A'),
  ('Châtelet', 'B'),
  ('Louvre', 'B'),
  'Louvre'],
 10)

## A*

In [4]:
def get_distance_estimate(current, goal):
    # Fix: Extract the first item if it's a tuple, otherwise keep the string
    station1 = current[0] if isinstance(current, tuple) else current
    station2 = goal[0] if isinstance(goal, tuple) else goal
    
    x1, y1 = STATION_COORDS[station1]
    x2, y2 = STATION_COORDS[station2]
    
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

In [5]:
def a_star_search(graph, start, goal):
    open_set = []
    heapq.heappush(open_set, (0, start))
    
    g_score = {node: float('inf') for node in graph}
    g_score[start] = 0
    
    came_from = {}
    
    while open_set:
        current_f, current = heapq.heappop(open_set)
        
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1] , g_score[goal]
            
        for neighbor, weight in graph.get(current, []):
            tentative_g = g_score[current] + weight
            
            if tentative_g < g_score.get(neighbor, float('inf')):
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                
                f_score = tentative_g + get_distance_estimate(neighbor, goal)
                heapq.heappush(open_set, (f_score, neighbor))

                
    return None

In [6]:
a_star_search(mini_graph, "Saint-Michel", "Louvre")

(['Saint-Michel',
  ('Saint-Michel', 'A'),
  ('Châtelet', 'A'),
  ('Châtelet', 'B'),
  ('Louvre', 'B'),
  'Louvre'],
 10)